# Phase 3: Data Assessment & Profiling

## Objective

The objective of this notebook is to perform an initial assessment of the NexaCart marketplace dataset before any data cleaning or business analysis.

This notebook focuses on understanding the structure, quality, and completeness of the available datasets by examining:

- Dataset inventory
- Dataset dimensions
- Column information
- Data types
- Missing values
- Duplicate records
- Unique values
- Basic statistical summary
- Initial data quality observations

The findings from this notebook will be used to guide the Data Cleaning and Feature Engineering phases.

In [2]:
import sys
print(sys.executable)

/Users/deepak/Desktop/Nexacart-Marketplace-Intelligence/.venv/bin/python


In [3]:
# ==========================================================
# Phase 3 : Data Assessment & Profiling
# Import Required Libraries
# ==========================================================

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np

from pathlib import Path

# Display Settings
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.float_format", "{:.2f}".format)

In [4]:
# ==========================================================
# Project Paths
# ==========================================================

PROJECT_ROOT = Path.cwd().parent

DATA_PATH = PROJECT_ROOT / "data" / "raw"

FILE_NAME = "NexaCart_Data.xlsx"

EXCEL_FILE = DATA_PATH / FILE_NAME

print(EXCEL_FILE)

/Users/deepak/Desktop/Nexacart-Marketplace-Intelligence/data/raw/NexaCart_Data.xlsx


In [5]:
if EXCEL_FILE.exists():
    print("Dataset found.")
else:
    raise FileNotFoundError(f"{EXCEL_FILE} not found.")

Dataset found.


In [6]:
# ==========================================================
# Load Excel Workbook
# ==========================================================

excel = pd.ExcelFile(EXCEL_FILE)

print("Workbook loaded successfully.")

Workbook loaded successfully.


In [7]:
# ==========================================================
# Available Sheets
# ==========================================================

sheet_names = excel.sheet_names

print(f"Total Sheets : {len(sheet_names)}\n")

for i, sheet in enumerate(sheet_names, start=1):
    print(f"{i}. {sheet}")

Total Sheets : 9

1. sellers_dataset
2. product_category_name_translati
3. products_dataset
4. order_reviews_dataset
5. orders_dataset
6. order_payments_dataset
7. order_items_dataset
8. geolocation_dataset
9. customers_dataset


In [8]:
# ==========================================================
# Load All Datasets
# ==========================================================

datasets = {
    sheet: pd.read_excel(EXCEL_FILE, sheet_name=sheet)
    for sheet in sheet_names
}

In [9]:
print(f"Datasets Loaded : {len(datasets)}\n")

for name in datasets:
    print(name)

Datasets Loaded : 9

sellers_dataset
product_category_name_translati
products_dataset
order_reviews_dataset
orders_dataset
order_payments_dataset
order_items_dataset
geolocation_dataset
customers_dataset


# Dataset Inventory

In this section, we generate a high-level summary of all datasets available in the NexaCart workbook.

The inventory includes:

- Dataset Name
- Number of Rows
- Number of Columns
- Memory Usage
- Column Names

This provides an overview of the data available for further assessment and analysis.

In [10]:
# ==========================================================
# Dataset Inventory
# ==========================================================

inventory = []

for dataset_name, df in datasets.items():

    inventory.append({
        "Dataset": dataset_name,
        "Rows": df.shape[0],
        "Columns": df.shape[1],
        "Memory (MB)": round(df.memory_usage(deep=True).sum() / (1024**2), 2)
    })

inventory_df = pd.DataFrame(inventory)

inventory_df

,Dataset,Rows,Columns,Memory (MB)
0,sellers_dataset,3095,4,0.66
1,product_category_name_translati,72,2,0.01
2,products_dataset,32951,9,6.79
3,order_reviews_dataset,99224,5,19.11
4,orders_dataset,99441,8,26.93
5,order_payments_dataset,103886,5,17.81
6,order_items_dataset,112650,7,32.12
7,geolocation_dataset,1000163,5,145.20
8,customers_dataset,99441,5,29.62


In [11]:
# ==========================================================
# Dataset Summary
# ==========================================================

for dataset_name, df in datasets.items():

    print("=" * 80)
    print(f"Dataset : {dataset_name}")
    print("=" * 80)

    print(f"Rows        : {df.shape[0]:,}")
    print(f"Columns     : {df.shape[1]}")
    print(f"Memory      : {round(df.memory_usage(deep=True).sum()/(1024**2),2)} MB")

    print("\nColumn Names:")

    for column in df.columns:
        print(f"• {column}")

    print("\n")

Dataset : sellers_dataset
Rows        : 3,095
Columns     : 4
Memory      : 0.66 MB

Column Names:
• seller_id
• seller_zip_code_prefix
• seller_city
• seller_state


Dataset : product_category_name_translati
Rows        : 72
Columns     : 2
Memory      : 0.01 MB

Column Names:
• Column1
• Column2


Dataset : products_dataset
Rows        : 32,951
Columns     : 9
Memory      : 6.79 MB

Column Names:
• product_id
• product_category_name
• product_name_lenght
• product_description_lenght
• product_photos_qty
• product_weight_g
• product_length_cm
• product_height_cm
• product_width_cm


Dataset : order_reviews_dataset
Rows        : 99,224
Columns     : 5
Memory      : 19.11 MB

Column Names:
• review_id
• order_id
• review_score
• review_creation_date
• review_answer_timestamp


Dataset : orders_dataset
Rows        : 99,441
Columns     : 8
Memory      : 26.93 MB

Column Names:
• order_id
• customer_id
• order_status
• order_purchase_timestamp
• order_approved_at
• order_delivered_carrier_

# Data Profiling

In this section, reusable profiling functions are created to perform a standardized assessment of every dataset.

Rather than writing repetitive code for each table, reusable functions improve code quality, maintainability, and scalability.

The profiling process examines:

- Dataset dimensions
- Data types
- Missing values
- Duplicate records
- Unique values
- Statistical summary
- Memory usage

These functions will be reused throughout the project whenever new datasets are introduced.

In [12]:
# ==========================================================
# Basic Dataset Information
# ==========================================================

def basic_info(df: pd.DataFrame) -> pd.DataFrame:
    """
    Returns high-level information about a dataset.
    """

    info = pd.DataFrame({
        "Column": df.columns,
        "Data Type": df.dtypes.values,
        "Non-Null Count": df.count().values,
        "Null Count": df.isna().sum().values,
        "Null %": (
            (df.isna().sum() / len(df)) * 100
        ).round(2).values,
        "Unique Values": df.nunique().values
    })

    return info

In [13]:
# ==========================================================
# Duplicate Analysis
# ==========================================================

def duplicate_summary(df: pd.DataFrame) -> dict:

    duplicates = df.duplicated().sum()

    return {
        "Total Rows": len(df),
        "Duplicate Rows": duplicates,
        "Duplicate %": round((duplicates / len(df)) * 100, 2)
    }

In [14]:
# ==========================================================
# Memory Usage
# ==========================================================

def memory_usage(df: pd.DataFrame) -> float:

    return round(
        df.memory_usage(deep=True).sum() / (1024**2),
        2
    )

In [15]:
# ==========================================================
# Numeric Summary
# ==========================================================

def numeric_summary(df: pd.DataFrame):

    numeric_df = df.select_dtypes(include=["number"])

    if numeric_df.empty:
        return None

    return numeric_df.describe().T

In [16]:
# ==========================================================
# Categorical Summary
# ==========================================================

def categorical_summary(df: pd.DataFrame):

    categorical_df = df.select_dtypes(include=["object"])

    if categorical_df.empty:
        return None

    summary = pd.DataFrame({
        "Unique Values": categorical_df.nunique(),
        "Most Frequent": categorical_df.mode().iloc[0],
        "Frequency": categorical_df.apply(lambda x: x.value_counts().iloc[0])
    })

    return summary

In [17]:
# ==========================================================
# Master Profiling Function
# ==========================================================

def profile_dataset(name: str, df: pd.DataFrame):

    print("=" * 80)
    print(f"DATASET : {name}")
    print("=" * 80)

    print(f"Rows      : {len(df):,}")
    print(f"Columns   : {df.shape[1]}")
    print(f"Memory    : {memory_usage(df)} MB")

    print("\nBasic Information")
    display(basic_info(df))

    print("\nDuplicate Summary")
    print(duplicate_summary(df))

    print("\nNumeric Summary")
    display(numeric_summary(df))

    print("\nCategorical Summary")
    display(categorical_summary(df))

In [18]:
for dataset_name, df in datasets.items():

    profile_dataset(dataset_name, df)

DATASET : sellers_dataset
Rows      : 3,095
Columns   : 4
Memory    : 0.66 MB

Basic Information


,Column,Data Type,Non-Null Count,Null Count,Null %,Unique Values
0,seller_id,str,3095,0,0.00,3095
1,seller_zip_code_prefix,int64,3095,0,0.00,2246
2,seller_city,str,3095,0,0.00,611
3,seller_state,str,3095,0,0.00,23



Duplicate Summary
{'Total Rows': 3095, 'Duplicate Rows': np.int64(0), 'Duplicate %': np.float64(0.0)}

Numeric Summary


,count,mean,std,min,25%,50%,75%,max
seller_zip_code_prefix,3095.00,32291.06,32713.45,1001.00,7093.50,14940.00,64552.50,99730.00



Categorical Summary


...

,Column,Data Type,Non-Null Count,Null Count,Null %,Unique Values
0,customer_id,str,99441,0,0.00,99441
1,customer_unique_id,str,99441,0,0.00,96096
2,customer_zip_code_prefix,int64,99441,0,0.00,14994
3,customer_city,str,99441,0,0.00,4119
4,customer_state,str,99441,0,0.00,27



Duplicate Summary
{'Total Rows': 99441, 'Duplicate Rows': np.int64(0), 'Duplicate %': np.float64(0.0)}

Numeric Summary


,count,mean,std,min,25%,50%,75%,max
customer_zip_code_prefix,99441.00,35137.47,29797.94,1003.00,11347.00,24416.00,58900.00,99990.00



Categorical Summary


,Unique Values,Most Frequent,Frequency
customer_id,99441,00012a2ce6f8dcda20d059ce98491703,1
customer_unique_id,96096,8d50f5eadf50201ccdcedfb9e2ac8455,17
customer_city,4119,sao paulo,15540
customer_state,27,SP,41746
